# MASLD-Bench · 01 — Extra Corpus Builders (Colab, Drive-synced edition)

Your local folders are synced to Google Drive via Drive for Desktop (folder name on Drive: **My Mac**). This notebook auto-discovers them whether they landed in `My Drive/My Mac/...` or a mirrored `My Drive/masld/...` layout, and builds the corpora that need more compute/RAM than the laptop (Tahoe perturbation, ArrayExpress panels).

**Already frozen locally — do not rebuild here:** MB-10xx/11xx (clinical train), MB-20xx (frozen test), MB-30xx (DNAm), MB-61xx/62xx/63xx (knowledge/drugs/affinity), MB-70xx (DGE), MB-8xxx (Fujiwara), MB-9xxx (KU test). All live in `masld-llm/prompts/` and are synced to Drive automatically.

Recipe: Zhavoronkov et al., *Cell* 189:5980–5994 (2026).

## 0 · Mount Drive and auto-discover the synced folders
Drive for Desktop syncs your Mac home as `My Drive/My Mac/...` (computers-sync) — Colab mounts it read-only, which is fine because all outputs below go to the writable masld-llm output dir.

In [1]:
from google.colab import drivedrive.mount('/content/drive')import os, glob# --- auto-discovery: handle both computers-sync and mirrored layouts ---SEARCH_ROOTS = ['/content/drive/Othercomputers/My Mac', '/content/drive/MyDrive']
# Drive for Desktop computers-sync mounts at Othercomputers/My Mac (read-only)
def find_dir(target_name, search_roots=None, depth=6):
    if search_roots is None: search_roots = SEARCH_ROOTS
    matches = []
    for root in search_roots:  # fast path: direct child
        direct = os.path.join(root, target_name)
        if os.path.isdir(direct): matches.append(direct)
    if matches: return matches
    def walk(path, d):
        if d > depth: return
        try:
            for entry in os.scandir(path):
                if entry.is_dir(follow_symlinks=False):
                    if entry.name == target_name: matches.append(entry.path)
                    walk(entry.path, d + 1)
        except (PermissionError, OSError): pass
    for root in search_roots:
        if os.path.exists(root): walk(root, 0)
    return matchescands = find_dir('masld-llm')          # the program home (has prompts/, scripts/)rev   = find_dir('masld-revision')     # source data (Kamzolas, Tahoe, etc.)git   = find_dir('masld-clinical-transcriptomics-github')print('masld-llm candidates:');           [print('  ', p) for p in cands]print('masld-revision candidates:');     [print('  ', p) for p in rev]print('github-folder candidates:');      [print('  ', p) for p in git]# pick the shallowest match of eachMASLD_LLM = min(cands, key=len) if cands else NoneMASLD_REV = min(rev, key=len) if rev else NoneMASLD_GIT = min(git, key=len) if git else None# outputs MUST go to a writable location: if masld-llm is read-only (computers-sync),# write to My Drive/masld-llm-outputs insteaddef is_writable(path):    try:        probe = os.path.join(path, '.write_test')        open(probe, 'w').close(); os.remove(probe)        return True    except Exception:        return Falseif MASLD_LLM and is_writable(MASLD_LLM):    OUT = os.path.join(MASLD_LLM, 'prompts')else:    OUT = '/content/drive/MyDrive/masld-llm-outputs/prompts'os.makedirs(OUT, exist_ok=True)print()print('MASLD_LLM =', MASLD_LLM)print('MASLD_REV =', MASLD_REV)print('MASLD_GIT =', MASLD_GIT)print('OUT (writable) =', OUT)

SyntaxError: invalid syntax (2489459175.py, line 1)

In [ ]:
# if auto-discovery missed anything, set paths manually:# MASLD_REV = '/content/drive/MyDrive/My Mac/subhadipbanerjee/masld-revision'# MASLD_LLM = '/content/drive/MyDrive/My Mac/subhadipbanerjee/masld-llm'assert MASLD_REV is not None, 'set MASLD_REV manually above'print('ready')

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 1 · Tahoe-100M perturbation builder (MB-91xx/92xx)
43 liver-duo DE shards (~4M rows each) — needs Colab RAM. Split by cell line: HCC-2998 + HCT15 held out for testing.

In [ ]:
import glob, random, json, gzip, hashlib, timeimport pyarrow.parquet as pqimport pandas as pdT0 = time.time()rng = random.Random(20260930)TEST_LINES = ['HCC-2998', 'HCT15']TARGET_N = 60_000shards = sorted(glob.glob(f'{MASLD_REV}/data/tahoe/pseudobulk_de/*.parquet'))print('shards found:', len(shards))assert shards, 'Tahoe shards not found - check MASLD_REV path'rows_train, rows_test, scanned = [], [], 0for shard in shards:    t = pq.read_table(shard, columns=['gene_name','log2FoldChange','padj','drug','Cell_Name_Vevo','concentration'])    df = t.to_pandas()    del t    scanned += len(df)    df = df[df['padj'] < 0.05]    is_test = df['Cell_Name_Vevo'].isin(TEST_LINES)    take = min(len(df[~is_test]), TARGET_N // max(1, len(shards)))    if take: rows_train.append(df[~is_test].sample(n=take, random_state=20260930))    if is_test.any(): rows_test.append(df[is_test].head(200))   # cap test side    del df    print(f'  {scanned:10d} rows scanned ({time.time()-T0:.0f}s)', flush=True)train = pd.concat(rows_train) if rows_train else pd.DataFrame()test  = pd.concat(rows_test) if rows_test else pd.DataFrame()print('train rows:', len(train), '| test rows:', len(test))def write_jsonl(records, path, meta):    meta = {'n_records': len(records), 'seed': 20260930, **meta}    with gzip.open(path, 'wt') as f:        f.write(json.dumps({'__meta__': meta}) + chr(10))        for r in records: f.write(json.dumps(r) + chr(10))    print('  wrote', path)recs = [dict(task='MB-9101', split='train', cell_line=r.Cell_Name_Vevo, drug=r.drug,             prompt=(f"In an in-vitro perturbation experiment, cell line {r.Cell_Name_Vevo} was treated with "                     f"{r.drug} at {r.concentration}. Does expression of {r.gene_name} increase or decrease "                     f"relative to control? Answer with increase or decrease."),             answer='increase' if r.log2FoldChange > 0 else 'decrease')        for r in train.itertuples()]write_jsonl(recs, f'{OUT}/MB-91xx_tahoe_train.jsonl.gz',            meta=dict(builder='Tahoe-100M liver shards', test_lines=TEST_LINES, n_scanned=scanned))recs = [dict(task='MB-9201', split='test', cell_line=r.Cell_Name_Vevo, drug=r.drug,             prompt=(f"In an in-vitro perturbation experiment, cell line {r.Cell_Name_Vevo} was treated with "                     f"{r.drug} at {r.concentration}. Does expression of {r.gene_name} increase or decrease "                     f"relative to control? Answer with increase or decrease."),             answer='increase' if r.log2FoldChange > 0 else 'decrease')        for r in test.itertuples()]write_jsonl(recs, f'{OUT}/MB-92xx_tahoe_test.jsonl.gz',            meta=dict(builder='Tahoe held-out cell lines', test_lines=TEST_LINES))print(f'done in {time.time()-T0:.0f}s')

## 2 · (Optional) ArrayExpress tier-1 panels
E-MTAB-15601/-15612/-15586/-15587/-15609/-15610 PHH spheroid perturbations. Download via BioStudies FTP into Colab (not synced — they're not in your folders), then apply the same builder pattern as §1. Deferred until Tahoe corpus is validated.

## 3 · Sync new corpora back into the local masld-llm/prompts/
If OUT is the mirrored masld-llm (writable), Drive sync delivers them to your laptop automatically. If OUT is masld-llm-outputs, copy the files into `~/masld-llm/prompts/` locally and update MANIFEST with the freeze-4 helper script (run locally).